# LeetCode 2260 — Minimum Consecutive Cards to Pick Up bằng Python

Link đề: https://leetcode.com/problems/minimum-consecutive-cards-to-pick-up/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def minimumCardPickup(self, cards: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển bài "đoạn liên tiếp ngắn nhất chứa hai lá bài trùng giá trị" thành "**khoảng cách nhỏ nhất giữa hai lần xuất hiện liên tiếp** của cùng một giá trị".
2. Dùng `dict` **lần xuất hiện gần nhất** (`last[x]`) — một trong những mẫu hash map quan trọng nhất.
3. Hiểu vì sao chỉ cần so với lần xuất hiện **gần nhất** (không cần mọi lần trước).
4. So sánh với cửa sổ trượt dùng `set` (co cửa sổ khi gặp trùng), gom nhóm chỉ số, và mảng thay `dict`.
5. Viết test đối chiếu với vét cạn trên mọi đoạn con; xử lý đúng trường hợp không có cặp trùng (`-1`).

## 2. Đề bài & ví dụ

Cho mảng `cards`, `cards[i]` là giá trị lá bài thứ `i`. Hai lá **khớp** nếu cùng giá trị.

Trả về **số lá liên tiếp ít nhất** phải nhặt (một đoạn con liên tiếp) để trong các lá đã nhặt có một cặp khớp; nếu không thể, trả `-1`.

**Ví dụ 1:** `cards = [3,4,2,3,4,7]` → `4` (`[3,4,2,3]` hoặc `[4,2,3,4]`).

**Ví dụ 2:** `cards = [1,0,5,3]` → `-1` (không có giá trị nào lặp lại).

**Ràng buộc:** `1 ≤ cards.length ≤ 10⁵`, `0 ≤ cards[i] ≤ 10⁶`.

In [1]:
from typing import List
from collections import defaultdict
import random, time

examples = [([3, 4, 2, 3, 4, 7], 4), ([1, 0, 5, 3], -1)]
for cards, exp in examples:
    print(cards, "-> kỳ vọng", exp)

[3, 4, 2, 3, 4, 7] -> kỳ vọng 4
[1, 0, 5, 3] -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 `dict` lưu lần xuất hiện gần nhất

Duyệt từ trái sang phải, `last[x] = i` sau khi xử lý `cards[i]`. Tại thời điểm xử lý `i`, `last[x]` (nếu có) là chỉ số **lớn nhất** `< i` có giá trị `x`.

In [2]:
last = {}
for i, x in enumerate([3, 4, 2, 3, 4, 7]):
    prev = last.get(x)
    print(f"i={i}, x={x}: lần gần nhất trước đó = {prev}")
    last[x] = i
print("cuối cùng:", last)

i=0, x=3: lần gần nhất trước đó = None
i=1, x=4: lần gần nhất trước đó = None
i=2, x=2: lần gần nhất trước đó = None
i=3, x=3: lần gần nhất trước đó = 0
i=4, x=4: lần gần nhất trước đó = 1
i=5, x=7: lần gần nhất trước đó = None
cuối cùng: {3: 3, 4: 4, 2: 2, 7: 5}


### 3.2 Độ dài đoạn `[a, b]` (bao gồm hai đầu) là `b − a + 1`

In [3]:
a, b = 0, 3
print("đoạn", list(range(a, b + 1)), "có", b - a + 1, "phần tử")

đoạn [0, 1, 2, 3] có 4 phần tử


### 3.3 `float("inf")` làm giá trị khởi tạo cho `min`

Khởi tạo `ans = inf`, cuối cùng `ans == inf` nghĩa là chưa tìm được cặp nào ⇒ `-1`.

In [4]:
ans = float("inf")
for v in [7, 3, 9]: ans = min(ans, v)
print(ans, float("inf") == float("inf"), -1 if float("inf") == float("inf") else 0)

3 True -1


### 3.4 `set` cho cửa sổ trượt

Cửa sổ `[l, r]` chứa các giá trị **phân biệt**; thêm `cards[r]`, nếu đã có thì có cặp trùng — co `l` lại cho tới khi hết trùng.

In [5]:
window = {3, 4, 2}
print(3 in window, 7 in window)
window.discard(3); print(window)

True False
{2, 4}


## 4. Ý tưởng nền tảng / trực giác

Một đoạn liên tiếp `[a, b]` chứa cặp khớp ⇔ có `i < j` trong `[a, b]` với `cards[i] = cards[j]`. Đoạn **ngắn nhất** chứa cặp `(i, j)` chính là `[i, j]` (độ dài `j − i + 1`). Vậy:

**Đáp án = min over mọi cặp `i < j` cùng giá trị của `(j − i + 1)`.**

Nhưng không cần xét mọi cặp: nếu giá trị `x` xuất hiện tại `p₁ < p₂ < p₃`, cặp `(p₁, p₃)` luôn dài hơn `(p₂, p₃)`. ⇒ chỉ cần các cặp **liên tiếp** trong danh sách vị trí của mỗi giá trị.

Tính tay ví dụ 1 `[3, 4, 2, 3, 4, 7]`:

| giá trị | vị trí | cặp liên tiếp | độ dài |
|---|---|---|---|
| 3 | 0, 3 | (0, 3) | 4 |
| 4 | 1, 4 | (1, 4) | 4 |
| 2 | 2 | — | — |
| 7 | 5 | — | — |

Min = 4. ✔

In [6]:
pos = defaultdict(list)
for i, x in enumerate([3, 4, 2, 3, 4, 7]): pos[x].append(i)
for x, p in pos.items():
    pairs = [(p[k], p[k + 1], p[k + 1] - p[k] + 1) for k in range(len(p) - 1)]
    print(f"giá trị {x}: vị trí {p}, cặp liên tiếp (i, j, độ dài) = {pairs}")

giá trị 3: vị trí [0, 3], cặp liên tiếp (i, j, độ dài) = [(0, 3, 4)]
giá trị 4: vị trí [1, 4], cặp liên tiếp (i, j, độ dài) = [(1, 4, 4)]
giá trị 2: vị trí [2], cặp liên tiếp (i, j, độ dài) = []
giá trị 7: vị trí [5], cặp liên tiếp (i, j, độ dài) = []


### 4.1 Minh họa "cặp xa hơn không bao giờ tốt hơn"

Với một giá trị xuất hiện ở các vị trí `p₁ < p₂ < … < p_k`, mọi cặp `(p_s, p_t)` với `t > s + 1` đều chứa cặp liên tiếp `(p_{t−1}, p_t)` bên trong, nên dài hơn. Ta liệt kê cả hai loại cho giá trị 5 trong `[5, 1, 5, 2, 5, 5]`:

In [7]:
arr = [5, 1, 5, 2, 5, 5]
p = [i for i, x in enumerate(arr) if x == 5]
all_pairs = [(p[s], p[t], p[t] - p[s] + 1) for s in range(len(p)) for t in range(s + 1, len(p))]
consecutive = [(p[s], p[s + 1], p[s + 1] - p[s] + 1) for s in range(len(p) - 1)]
print("mọi cặp      :", all_pairs, "-> min", min(x[2] for x in all_pairs))
print("cặp liên tiếp:", consecutive, "-> min", min(x[2] for x in consecutive))
print("số cặp phải xét giảm từ", len(all_pairs), "xuống", len(consecutive))

mọi cặp      : [(0, 2, 3), (0, 4, 5), (0, 5, 6), (2, 4, 3), (2, 5, 4), (4, 5, 2)] -> min 2
cặp liên tiếp: [(0, 2, 3), (2, 4, 3), (4, 5, 2)] -> min 2
số cặp phải xét giảm từ 6 xuống 3


Với giá trị xuất hiện `k` lần, số cặp giảm từ `k(k−1)/2` xuống `k − 1` — đó là bước đột phá từ O(n²) xuống O(n).

## 5. Cách 1 — Brute Force / baseline: thử mọi đoạn con

**Ý tưởng:** với mỗi điểm đầu `a`, mở rộng điểm cuối `b` và giữ một `set` các giá trị; lần đầu gặp giá trị đã có ⇒ đoạn `[a, b]` là đoạn ngắn nhất bắt đầu từ `a` có cặp trùng ⇒ cập nhật đáp án và `break`.

In [8]:
def brute(cards):
    n = len(cards); best = None
    for a in range(n):
        seen = set()
        for b in range(a, n):
            if cards[b] in seen:
                L = b - a + 1
                if best is None or L < best: best = L
                break
            seen.add(cards[b])
    return -1 if best is None else best

for cards, exp in examples:
    got = brute(cards)
    print(got, "OK" if got == exp else "SAI")

4 OK
-1 OK


**Độ phức tạp:** worst O(n²) (khi không có hoặc ít cặp trùng, mỗi `a` quét tới cuối). `n = 10⁵` ⇒ 10¹⁰ ⇒ TLE.

### 5.1 Baseline "rất thô": mọi cặp `(i, j)`

O(n²) cặp — để kiểm chứng định nghĩa "min `j − i + 1` qua mọi cặp trùng".

In [9]:
def brute_pairs(cards):
    n = len(cards)
    cands = [j - i + 1 for i in range(n) for j in range(i + 1, n) if cards[i] == cards[j]]
    return min(cands) if cands else -1

for cards, exp in examples:
    print(brute_pairs(cards), "OK" if brute_pairs(cards) == exp else "SAI")

4 OK
-1 OK


**Nút thắt:** với mỗi `j` ta chỉ cần **một** `i` — lần xuất hiện gần nhất trước `j` của cùng giá trị. Lưu nó vào `dict` ⇒ O(1) mỗi bước.

In [10]:
for n in [2000, 5000]:
    a = list(range(n))                     # không có cặp trùng: worst case
    t = time.perf_counter(); brute(a); print(f"n={n}: {(time.perf_counter()-t)*1000:.0f} ms")

n=2000: 100 ms


n=5000: 515 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** với mỗi chỉ số `j`, đoạn ngắn nhất **kết thúc tại `j`** và có cặp trùng mà `j` tham gia là `[last[x], j]` với `x = cards[j]` và `last[x]` = lần xuất hiện gần nhất trước `j`. Đáp án = min qua mọi `j`.

Quy tắc khi duyệt `j`:
- **Nếu** `x = cards[j]` đã có trong `last` **thì** `ans = min(ans, j − last[x] + 1)`.
- **Luôn** cập nhật `last[x] = j` (để lần sau so với vị trí gần nhất).

**Bất biến:** trước khi xử lý `j`, với mọi giá trị `v`, `last[v]` = chỉ số lớn nhất `< j` có giá trị `v` (nếu có); `ans` = min độ dài qua mọi cặp liên tiếp cùng giá trị nằm trong `cards[0..j−1]`.

**Cận dưới hữu ích:** đáp án ≥ 2; nếu gặp hai lá **kề nhau** bằng nhau thì có thể dừng ngay (đáp án 2).

**Nguyên lý chuồng bồ câu:** nếu `n > (số giá trị phân biệt có thể)` thì chắc chắn có cặp trùng. Với giá trị `0..10⁶` điều này không giúp nhiều, nhưng ở biến thể bảng chữ cái (26 chữ) thì đáp án luôn ≤ 27.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `dict` lần xuất hiện gần nhất (cách chuẩn)

In [11]:
def last_seen(cards):
    last = {}; ans = float("inf")
    for i, x in enumerate(cards):
        if x in last:
            ans = min(ans, i - last[x] + 1)
        last[x] = i
    return -1 if ans == float("inf") else ans

for cards, exp in examples:
    got = last_seen(cards)
    print(got, "OK" if got == exp else "SAI")

4 OK
-1 OK


**Độ phức tạp cách 2:** O(n) thời gian trung bình, O(số giá trị phân biệt) bộ nhớ.

### 7.2 Cách 3 — Cửa sổ trượt với `set` (co khi gặp trùng)

Giữ cửa sổ `[l, r]` gồm các giá trị phân biệt. Khi thêm `cards[r]` mà đã có trong cửa sổ: cặp trùng là `cards[r]` với lần xuất hiện **trong cửa sổ** — đó chính là lần gần nhất. Cách làm: co `l` (bỏ dần các giá trị khỏi `set`) cho tới khi `cards[l] == cards[r]` — lúc đó `l` chính là vị trí bản cũ, độ dài đoạn trùng là `r − l + 1`; sau đó tăng `l` thêm một để bỏ bản cũ.

In [12]:
def sliding_window(cards):
    window = set(); l = 0; ans = float("inf")
    for r, x in enumerate(cards):
        if x in window:
            while cards[l] != x:            # bỏ các phần tử trước bản cũ của x
                window.discard(cards[l]); l += 1
            ans = min(ans, r - l + 1)       # cards[l] == x là bản cũ
            l += 1                          # bỏ bản cũ (bản mới ở r vẫn trong cửa sổ)
        else:
            window.add(x)
    return -1 if ans == float("inf") else ans

for cards, exp in examples:
    got = sliding_window(cards)
    print(got, "OK" if got == exp else "SAI")

4 OK
-1 OK


Mỗi chỉ số vào/ra cửa sổ tối đa một lần ⇒ **O(n)**. Lưu ý tinh tế: khi bỏ bản cũ của `x`, ta **không** `discard(x)` khỏi `set` vì bản mới ở `r` vẫn nằm trong cửa sổ.

### 7.3 Cách 4 — Gom vị trí theo giá trị, lấy hiệu kề nhau

In [13]:
def group_positions(cards):
    pos = defaultdict(list)
    for i, x in enumerate(cards): pos[x].append(i)
    ans = float("inf")
    for p in pos.values():
        for a, b in zip(p, p[1:]):
            ans = min(ans, b - a + 1)
    return -1 if ans == float("inf") else ans

for cards, exp in examples:
    print(group_positions(cards), "OK" if group_positions(cards) == exp else "SAI")

4 OK
-1 OK


**Độ phức tạp cách 4:** O(n) thời gian, O(n) bộ nhớ (lưu mọi vị trí).

### 7.4 Cách 5 — Mảng thay `dict` (giá trị ≤ 10⁶)

`last = [-1] * (max(cards) + 1)`: truy cập mảng nhanh hơn băm; tốn tối đa ~8 MB.

In [14]:
def array_last(cards):
    last = [-1] * (max(cards) + 1); ans = float("inf")
    for i, x in enumerate(cards):
        if last[x] >= 0:
            d = i - last[x] + 1
            if d < ans:
                ans = d
                if ans == 2: return 2          # không thể ngắn hơn
        last[x] = i
    return -1 if ans == float("inf") else ans

for cards, exp in examples:
    print(array_last(cards), "OK" if array_last(cards) == exp else "SAI")

4 OK
-1 OK


### 7.5 Cách 6 — Sắp xếp cặp (giá trị, chỉ số)

Sắp `(cards[i], i)`; các bản cùng giá trị nằm liền nhau theo chỉ số tăng ⇒ hiệu chỉ số của hai phần tử kề nhau cùng giá trị. O(n log n), không cần băm.

In [15]:
def sort_pairs(cards):
    a = sorted((x, i) for i, x in enumerate(cards)); ans = float("inf")
    for (x1, i1), (x2, i2) in zip(a, a[1:]):
        if x1 == x2: ans = min(ans, i2 - i1 + 1)
    return -1 if ans == float("inf") else ans

for cards, exp in examples:
    print(sort_pairs(cards), "OK" if sort_pairs(cards) == exp else "SAI")

4 OK
-1 OK


## 8. TRACE từng bước

### 8.1 Trace cách 2 trên ví dụ 1

| i | x | last[x] trước | độ dài ứng viên | ans | last sau |
|---|---|---|---|---|---|
| 0 | 3 | — | — | ∞ | {3:0} |
| 1 | 4 | — | — | ∞ | {3:0, 4:1} |
| 2 | 2 | — | — | ∞ | {…, 2:2} |
| 3 | 3 | 0 | 3 − 0 + 1 = 4 | 4 | {3:3, …} |
| 4 | 4 | 1 | 4 | 4 | {4:4, …} |
| 5 | 7 | — | — | 4 | {…, 7:5} |

In [16]:
def trace_last(cards):
    last = {}; ans = float("inf")
    for i, x in enumerate(cards):
        if x in last:
            cand = i - last[x] + 1; ans = min(ans, cand)
            print(f"i={i}, x={x}: last[{x}]={last[x]} -> đoạn [{last[x]}, {i}] dài {cand}, ans={ans}")
        else:
            print(f"i={i}, x={x}: lần đầu gặp")
        last[x] = i
    return -1 if ans == float("inf") else ans

print("Kết quả:", trace_last([3, 4, 2, 3, 4, 7]))

i=0, x=3: lần đầu gặp
i=1, x=4: lần đầu gặp
i=2, x=2: lần đầu gặp
i=3, x=3: last[3]=0 -> đoạn [0, 3] dài 4, ans=4
i=4, x=4: last[4]=1 -> đoạn [1, 4] dài 4, ans=4
i=5, x=7: lần đầu gặp
Kết quả: 4


### 8.2 Trace một giá trị xuất hiện nhiều lần: `[5, 1, 5, 2, 2, 5]`

Giá trị 5 ở vị trí 0, 2, 5: cặp `(0, 2)` dài 3, cặp `(2, 5)` dài 4 — cặp `(0, 5)` không cần xét. Giá trị 2 ở 3, 4 ⇒ dài 2 (tốt nhất).

In [17]:
print("Kết quả:", trace_last([5, 1, 5, 2, 2, 5]))

i=0, x=5: lần đầu gặp
i=1, x=1: lần đầu gặp
i=2, x=5: last[5]=0 -> đoạn [0, 2] dài 3, ans=3
i=3, x=2: lần đầu gặp
i=4, x=2: last[2]=3 -> đoạn [3, 4] dài 2, ans=2
i=5, x=5: last[5]=2 -> đoạn [2, 5] dài 4, ans=2
Kết quả: 2


### 8.3 Trace cách 3 (cửa sổ trượt)

In [18]:
def trace_window(cards):
    window = set(); l = 0; ans = float("inf")
    for r, x in enumerate(cards):
        if x in window:
            while cards[l] != x:
                window.discard(cards[l]); l += 1
            ans = min(ans, r - l + 1)
            print(f"r={r}, x={x}: trùng với vị trí {l} -> độ dài {r - l + 1}, ans={ans}")
            l += 1
        else:
            window.add(x)
        print(f"     cửa sổ [{l}, {r}] = {cards[l:r + 1]}, set = {sorted(window)}")
    return ans

trace_window([5, 1, 5, 2, 2, 5])

     cửa sổ [0, 0] = [5], set = [5]
     cửa sổ [0, 1] = [5, 1], set = [1, 5]
r=2, x=5: trùng với vị trí 0 -> độ dài 3, ans=3
     cửa sổ [1, 2] = [1, 5], set = [1, 5]
     cửa sổ [1, 3] = [1, 5, 2], set = [1, 2, 5]
r=4, x=2: trùng với vị trí 3 -> độ dài 2, ans=2
     cửa sổ [4, 4] = [2], set = [2]
     cửa sổ [4, 5] = [2, 5], set = [2, 5]


2

## 9. Code nộp LeetCode

In [19]:
from typing import List

class Solution:
    def minimumCardPickup(self, cards: List[int]) -> int:
        last = {}                       # giá trị -> chỉ số xuất hiện gần nhất
        ans = float("inf")
        for i, x in enumerate(cards):
            if x in last:
                ans = min(ans, i - last[x] + 1)
            last[x] = i                 # luôn cập nhật về vị trí mới nhất
        return -1 if ans == float("inf") else ans

In [20]:
sol = Solution()
for cards, exp in examples:
    got = sol.minimumCardPickup(cards)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

4 | kỳ vọng 4 | OK
-1 | kỳ vọng -1 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1.** Đáp án = `min { j − i + 1 : i < j, cards[i] = cards[j] }` (hoặc −1 nếu tập rỗng).
- Một đoạn `[a, b]` có cặp khớp `(i, j)` thì `a ≤ i < j ≤ b` ⇒ độ dài ≥ `j − i + 1`.
- Ngược lại đoạn `[i, j]` có cặp khớp và độ dài đúng `j − i + 1`.

**Bổ đề 2.** Chỉ cần các cặp `(i, j)` mà `i` là lần xuất hiện **gần nhất** trước `j`. Nếu `cards[i] = cards[j]` và có `i < k < j` với `cards[k] = cards[j]` thì `j − k + 1 < j − i + 1`, nên cặp `(i, j)` không bao giờ là tối ưu duy nhất.

**Thuật toán** xét đúng mọi cặp ở Bổ đề 2 (mỗi `j` với `last[cards[j]]`) theo bất biến ở mục 6 ⇒ đúng. ∎

## 11. Phân tích Time / Space Complexity

`n ≤ 10⁵`, `V = max(cards) ≤ 10⁶`, `D` = số giá trị phân biệt.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi đoạn (break sớm) | O(n²) | O(n) |
| 1b. Mọi cặp | O(n²) | O(1) |
| 2. dict lần gần nhất | O(n) | O(D) |
| 3. Cửa sổ trượt + set | O(n) | O(D) |
| 4. Gom vị trí | O(n) | O(n) |
| 5. Mảng lần gần nhất | O(n + V) | O(V) |
| 6. Sắp xếp cặp | O(n log n) | O(n) |

In [21]:
rng = random.Random(2260)
tests = {"không trùng": list(range(10**5)), "ngẫu nhiên 1e6": [rng.randint(0, 10**6) for _ in range(10**5)]}
for tname, data in tests.items():
    res = []
    for name, f in [("last_seen", last_seen), ("sliding_window", sliding_window), ("group_positions", group_positions),
                    ("array_last", array_last), ("sort_pairs", sort_pairs), ("Solution", Solution().minimumCardPickup)]:
        t = time.perf_counter(); res.append(f(data)); dt = time.perf_counter() - t
        print(f"{tname:<15} {name:<16} -> {res[-1]:>5}  {dt*1000:6.1f} ms")
    assert len(set(res)) == 1

không trùng     last_seen        ->    -1    11.0 ms
không trùng     sliding_window   ->    -1     6.2 ms
không trùng     group_positions  ->    -1    36.0 ms
không trùng     array_last       ->    -1     5.3 ms
không trùng     sort_pairs       ->    -1    39.8 ms
không trùng     Solution         ->    -1     6.0 ms
ngẫu nhiên 1e6  last_seen        ->     8    10.8 ms


ngẫu nhiên 1e6  sliding_window   ->     8    11.4 ms


ngẫu nhiên 1e6  group_positions  ->     8    39.9 ms


ngẫu nhiên 1e6  array_last       ->     8    20.4 ms


ngẫu nhiên 1e6  sort_pairs       ->     8    63.0 ms
ngẫu nhiên 1e6  Solution         ->     8    12.7 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [22]:
ALL = {"brute": brute, "brute_pairs": brute_pairs, "last_seen": last_seen, "sliding_window": sliding_window,
       "group_positions": group_positions, "array_last": array_last, "sort_pairs": sort_pairs,
       "Solution": Solution().minimumCardPickup}
edge = examples + [
    ([7], -1),
    ([7, 7], 2),
    ([1, 2, 1], 3),
    ([1, 2, 3, 1, 2, 3], 4),
    ([0, 0, 0, 0], 2),
    ([5, 1, 5, 2, 2, 5], 2),
    ([10**6, 0, 10**6], 3),
    ([1, 2, 3, 4, 5, 1], 6),           # cặp duy nhất ở hai đầu
]
for cards, exp in edge:
    for name, f in ALL.items():
        a = list(cards)
        assert f(a) == exp, (name, cards, f(list(cards)), exp)
        assert a == cards
print("Edge case OK:", len(edge))

Edge case OK: 10


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [23]:
rng = random.Random(42)
for _ in range(3000):
    n = rng.randint(1, 30); hi = rng.choice([1, 3, 10, 30, 10**6])
    cards = [rng.randint(0, hi) for _ in range(n)]
    ref = brute_pairs(cards)
    for name, f in ALL.items():
        assert f(list(cards)) == ref, (name, cards)
print("3000 case ngẫu nhiên OK")

3000 case ngẫu nhiên OK


### 12.3 Tính chất: đáp án ≤ (số giá trị phân biệt) + 1 (chuồng bồ câu); đảo ngược mảng không đổi đáp án

In [24]:
rng = random.Random(7)
for _ in range(500):
    cards = [rng.randint(0, 6) for _ in range(rng.randint(1, 60))]
    a = Solution().minimumCardPickup(cards)
    assert a == Solution().minimumCardPickup(cards[::-1])
    if a != -1:
        assert 2 <= a <= len(set(cards)) + 1
    else:
        assert len(set(cards)) == len(cards)
print("tính chất OK")

tính chất OK


Vì sao `≤ D + 1`? Một đoạn dài `D + 1` chứa `D + 1` lá nhưng chỉ có `D` giá trị khác nhau ⇒ có hai lá trùng (nguyên lý chuồng bồ câu). Ngược lại đáp án `-1` ⇔ mọi giá trị phân biệt.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ lưu lần xuất hiện **đầu tiên** (không cập nhật `last`)

In [25]:
def wrong_first_only(cards):
    first = {}; ans = float("inf")
    for i, x in enumerate(cards):
        if x in first:
            ans = min(ans, i - first[x] + 1)
        else:
            first[x] = i                        # SAI: không cập nhật về vị trí mới nhất
    return -1 if ans == float("inf") else ans

print("bản sai :", wrong_first_only([5, 1, 2, 3, 5, 5]), "| bản đúng:", Solution().minimumCardPickup([5, 1, 2, 3, 5, 5]))

bản sai : 5 | bản đúng: 2


**Vì sao sai:** ở `[5, 1, 2, 3, 5, 5]`, lần cuối (chỉ số 5) bị so với chỉ số 0 (dài 6) thay vì chỉ số 4 (dài 2). **Sửa:** luôn gán `last[x] = i`.

### Lỗi 2 — Quên `+ 1` (trả khoảng cách thay vì số lá)

In [26]:
def wrong_no_plus_one(cards):
    last = {}; ans = float("inf")
    for i, x in enumerate(cards):
        if x in last: ans = min(ans, i - last[x])
        last[x] = i
    return -1 if ans == float("inf") else ans

print("bản sai :", wrong_no_plus_one([3, 4, 2, 3, 4, 7]), "| bản đúng:", Solution().minimumCardPickup([3, 4, 2, 3, 4, 7]))

bản sai : 3 | bản đúng: 4


**Sửa:** số lá trong đoạn `[i, j]` là `j − i + 1`.

### Lỗi 3 — Cập nhật `last[x]` **trước** khi so sánh

In [27]:
def wrong_update_first(cards):
    last = {}; ans = float("inf")
    for i, x in enumerate(cards):
        last[x] = i                               # SAI: ghi đè trước
        if x in last: ans = min(ans, i - last[x] + 1)
    return -1 if ans == float("inf") else ans

print("bản sai :", wrong_update_first([1, 0, 5, 3]), "| bản đúng:", Solution().minimumCardPickup([1, 0, 5, 3]))

bản sai : 1 | bản đúng: -1


**Vì sao sai:** sau khi ghi đè, `last[x] == i` ⇒ luôn có "cặp" độ dài 1 với chính nó. **Sửa:** so sánh trước, cập nhật sau.

### Lỗi 4 — Trả về `inf` (hoặc số rất lớn) thay vì `-1`

In [28]:
def wrong_no_minus_one(cards):
    last = {}; ans = float("inf")
    for i, x in enumerate(cards):
        if x in last: ans = min(ans, i - last[x] + 1)
        last[x] = i
    return ans                                    # SAI

print("bản sai :", wrong_no_minus_one([1, 0, 5, 3]), "| bản đúng:", Solution().minimumCardPickup([1, 0, 5, 3]))

bản sai : inf | bản đúng: -1


**Sửa:** `return -1 if ans == inf else ans`. (Trên LeetCode, trả `float` còn sai kiểu dữ liệu.)

### Lỗi 5 — Cửa sổ trượt: `discard(x)` khi loại bản cũ

In [29]:
def wrong_window_discard(cards):
    window = set(); l = 0; ans = float("inf")
    for r, x in enumerate(cards):
        if x in window:
            while cards[l] != x:
                window.discard(cards[l]); l += 1
            ans = min(ans, r - l + 1)
            window.discard(x); l += 1              # SAI: bản mới ở r cũng bị xóa khỏi set
        else:
            window.add(x)
    return -1 if ans == float("inf") else ans

c = [5, 9, 5, 5]
print("bản sai :", wrong_window_discard(c), "| bản đúng:", Solution().minimumCardPickup(c))
print("bản sai :", wrong_window_discard([7, 1, 7, 2, 7]), "| bản đúng:", Solution().minimumCardPickup([7, 1, 7, 2, 7]))

bản sai : 3 | bản đúng: 2
bản sai : 3 | bản đúng: 3


**Vì sao sai:** sau khi loại bản cũ, bản mới `cards[r]` vẫn nằm trong cửa sổ — nếu xóa khỏi `set`, lần xuất hiện tiếp theo của `x` sẽ không bị phát hiện là trùng. Ở `[5, 9, 5, 5]` cặp `(2, 3)` dài 2 bị bỏ sót. **Sửa:** chỉ tăng `l`, không `discard(x)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đoạn liên tiếp ngắn nhất chứa hai phần tử bằng nhau" ⇒ **khoảng cách nhỏ nhất giữa hai lần xuất hiện liên tiếp** ⇒ `dict` lần gần nhất.
- Mẫu "so sánh trước, cập nhật sau" với `last[x]` dùng ở rất nhiều bài: 1 Two Sum, 219 Contains Duplicate II, 3 Longest Substring Without Repeating Characters, 1679.
- Chỉ cần lần xuất hiện **gần nhất** vì cặp xa hơn luôn dài hơn.
- Giá trị nhỏ ⇒ mảng thay `dict`; dừng sớm khi đạt cận dưới (2).
- Bài cùng họ: 219 Contains Duplicate II (khoảng cách ≤ k), 2190, 1624 Largest Substring Between Two Equal Characters (khoảng cách **lớn** nhất — dùng lần xuất hiện **đầu tiên**!).

## 15. Biến thể của bài

### 15.1 Biến thể A — LC 1624: khoảng cách **lớn nhất** giữa hai ký tự bằng nhau (không tính hai đầu)

Ngược lại: lưu lần xuất hiện **đầu tiên** (không cập nhật).

In [30]:
def max_length_between_equal(s):
    first = {}; best = -1
    for i, ch in enumerate(s):
        if ch in first: best = max(best, i - first[ch] - 1)
        else: first[ch] = i
    return best

print(max_length_between_equal("aa"), max_length_between_equal("abca"), max_length_between_equal("cbzxy"))

0 2 -1


Đây chính là "Lỗi 1" của bài gốc nhưng lại đúng cho bài này — chọn **đầu tiên** hay **gần nhất** tùy câu hỏi là min hay max.

### 15.2 Biến thể B — Cần **k** lá cùng giá trị (không chỉ 2)

Với mỗi giá trị, giữ danh sách vị trí; đoạn ngắn nhất chứa `k` bản là `pos[t + k − 1] − pos[t] + 1` (cửa sổ trượt dài `k` trên danh sách vị trí).

In [31]:
def min_pickup_k(cards, k):
    pos = defaultdict(list)
    for i, x in enumerate(cards): pos[x].append(i)
    best = float("inf")
    for p in pos.values():
        for t in range(len(p) - k + 1):
            best = min(best, p[t + k - 1] - p[t] + 1)
    return -1 if best == float("inf") else best

def min_pickup_k_brute(cards, k):
    n = len(cards); best = -1
    for a in range(n):
        for b in range(a, n):
            seg = cards[a:b + 1]
            if any(seg.count(v) >= k for v in set(seg)):
                if best == -1 or b - a + 1 < best: best = b - a + 1
                break
    return best

print(min_pickup_k([3, 4, 3, 2, 3, 4, 4], 3), min_pickup_k([3, 4, 2, 3, 4, 7], 2))
rng = random.Random(152)
for _ in range(400):
    c = [rng.randint(0, 3) for _ in range(rng.randint(1, 15))]; k = rng.randint(1, 4)
    assert min_pickup_k(c, k) == min_pickup_k_brute(c, k)
print("khớp brute force")

5 4
khớp brute force


### 15.3 Biến thể C — LC 219: có cặp trùng với khoảng cách chỉ số ≤ k không?

Cùng `dict` lần gần nhất, chỉ đổi điều kiện.

In [32]:
def contains_nearby_duplicate(nums, k):
    last = {}
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k: return True
        last[x] = i
    return False

print(contains_nearby_duplicate([1, 2, 3, 1], 3), contains_nearby_duplicate([1, 2, 3, 1, 2, 3], 2))

True False


Liên hệ: `contains_nearby_duplicate(nums, k)` là `True` ⇔ `minimumCardPickup(nums) ≤ k + 1`.

In [33]:
rng = random.Random(153)
for _ in range(1000):
    c = [rng.randint(0, 8) for _ in range(rng.randint(1, 20))]; k = rng.randint(0, 10)
    a = Solution().minimumCardPickup(c)
    assert contains_nearby_duplicate(c, k) == (a != -1 and a <= k + 1)
print("quan hệ giữa LC 219 và LC 2260: đúng")

quan hệ giữa LC 219 và LC 2260: đúng


### 15.4 Biến thể D — Bài **vòng tròn** (lá cuối kề lá đầu)

Đoạn liên tiếp có thể vắt qua cuối–đầu. Ngoài các cặp liên tiếp thông thường, mỗi giá trị còn có cặp "vòng": lần xuất hiện **cuối** `q` và **đầu** `p`, độ dài `n − q + p + 1`. (Nếu giá trị chỉ xuất hiện một lần thì không có cặp.)

In [34]:
def min_pickup_circular(cards):
    n = len(cards); first, last = {}, {}; best = float("inf")
    for i, x in enumerate(cards):
        if x in last: best = min(best, i - last[x] + 1)
        else: first[x] = i
        last[x] = i
    for x in first:
        if first[x] != last[x]:
            best = min(best, n - last[x] + first[x] + 1)
    return -1 if best == float("inf") else best

def min_pickup_circular_brute(cards):
    n = len(cards); best = -1
    for a in range(n):
        seen = set()
        for L in range(1, n + 1):
            v = cards[(a + L - 1) % n]
            if v in seen:
                if best == -1 or L < best: best = L
                break
            seen.add(v)
    return best

print(min_pickup_circular([1, 2, 3, 4, 1]), "(thẳng:", Solution().minimumCardPickup([1, 2, 3, 4, 1]), ")")
rng = random.Random(154)
for _ in range(800):
    c = [rng.randint(0, 5) for _ in range(rng.randint(1, 12))]
    assert min_pickup_circular(c) == min_pickup_circular_brute(c)
print("vòng tròn: khớp brute force")

2 (thẳng: 5 )
vòng tròn: khớp brute force


Ở `[1, 2, 3, 4, 1]`, hai số 1 ở hai đầu: bài thẳng cần cả 5 lá, bài vòng chỉ cần 2 lá (lá cuối và lá đầu kề nhau).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi đoạn | mở rộng tới khi trùng | O(n²) | O(n) | kiểm chứng |
| 2. dict lần gần nhất | so với `last[x]` | O(n) | O(D) | **mặc định** |
| 3. Cửa sổ trượt | set, co khi trùng | O(n) | O(D) | khi tư duy theo cửa sổ |
| 4. Gom vị trí | hiệu kề nhau | O(n) | O(n) | biến thể k bản |
| 5. Mảng | không băm, dừng sớm | O(n + V) | O(V) | giá trị nhỏ |
| 6. Sắp xếp cặp | (giá trị, chỉ số) | O(n log n) | O(n) | không dùng băm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumCardPickup([1, 2, 3, 2, 1])` và `minimumCardPickup([4, 4])`?

**Bài 2 (tính tay):** Trace cách 2 cho `[9, 8, 7, 9, 8, 9]`; ở mỗi bước ghi `last` và `ans`.

**Bài 3 (điền TODO):** Viết `bt3_todo(cards)` trả về **đoạn** `[l, r]` ngắn nhất (l nhỏ nhất nếu hòa) hoặc `None`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(cards)` cài cách 3 (cửa sổ trượt) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(cards, queries)` trả lời nhiều truy vấn `(l, r)`: đáp án của bài gốc trên mảng con `cards[l..r]`. Gợi ý: tiền xử lý `prev[j]` = lần xuất hiện gần nhất trước `j` của `cards[j]`; truy vấn = min `j − prev[j] + 1` với `l ≤ prev[j]`, `j ≤ r` (cài O(độ dài đoạn) mỗi truy vấn).

In [35]:
def bt3_todo(cards):
    # TODO: đoạn [l, r] ngắn nhất hoặc None
    pass

def bt4_todo(cards):
    # TODO: cửa sổ trượt
    pass

def bt5_todo(cards, queries):
    # TODO: truy vấn trên mảng con
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tìm các cặp trùng. Mức 2: `2` ở 1 và 3 (dài 3), `1` ở 0 và 4 (dài 5). Mức 3: `3` và `2`.

**Bài 2.** Mức 1: `i = 3` gặp lại 9 (last 0) ⇒ 4. Mức 2: `i = 4` gặp lại 8 (last 1) ⇒ 4. Mức 3: `i = 5` gặp lại 9 (last 3) ⇒ 3 ⇒ đáp án 3.

**Bài 3.** Mức 1: như cách 2, lưu thêm `(last[x], i)` khi cập nhật min. Mức 2: dùng `<` để giữ đoạn đầu tiên. Mức 3: nếu hòa độ dài, đoạn tìm thấy trước có `r` nhỏ hơn ⇒ `l` cũng nhỏ hơn (cùng độ dài).

**Bài 4.** Mức 1: `set` + con trỏ `l`. Mức 2: khi trùng, co tới bản cũ. Mức 3: không xóa `x` khỏi set.

**Bài 5.** Mức 1: tính `prev` một lần. Mức 2: với truy vấn, duyệt `j` trong `[l, r]`. Mức 3: điều kiện `prev[j] ≥ l`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `3`, `2`. **Bài 2:** `ans` lần lượt ∞, ∞, ∞, 4, 4, 3 ⇒ `3`.

In [36]:
assert Solution().minimumCardPickup([1, 2, 3, 2, 1]) == 3
assert Solution().minimumCardPickup([4, 4]) == 2
assert Solution().minimumCardPickup([9, 8, 7, 9, 8, 9]) == 3

def bt3(cards):
    last = {}; best = None
    for i, x in enumerate(cards):
        if x in last and (best is None or i - last[x] < best[1] - best[0]):
            best = (last[x], i)
        last[x] = i
    return None if best is None else list(best)

def bt4(cards):
    return sliding_window(cards)

def bt5(cards, queries):
    last = {}; prev = []
    for i, x in enumerate(cards):
        prev.append(last.get(x, -1)); last[x] = i
    out = []
    for l, r in queries:
        best = min((j - prev[j] + 1 for j in range(l, r + 1) if prev[j] >= l), default=-1)
        out.append(best)
    return out

def bt3_brute(cards):
    n = len(cards)
    for L in range(2, n + 1):
        for a in range(n - L + 1):
            if len(set(cards[a:a + L])) < L: return [a, a + L - 1]
    return None

rng = random.Random(1960)
for _ in range(1500):
    c = [rng.randint(0, 6) for _ in range(rng.randint(1, 20))]
    assert bt3(c) == bt3_brute(c), c
    assert bt4(c) == brute(c)
    qs = []
    for _ in range(3):
        l = rng.randrange(len(c)); r = rng.randrange(l, len(c)); qs.append((l, r))
    assert bt5(c, qs) == [brute(c[l:r + 1]) for l, r in qs]
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 3: đoạn ngắn nhất "tìm thấy đầu tiên khi quét `i` tăng dần" có `r` nhỏ nhất trong các đoạn cùng độ dài, do đó `l = r − độ dài + 1` cũng nhỏ nhất — assert so với vét cạn theo độ dài tăng dần xác nhận điều này.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
last = {}; ans = inf
for i, x in enumerate(cards):
    if x in last:
        ans = min(ans, i - last[x] + 1)   # so sánh TRƯỚC
    last[x] = i                           # cập nhật SAU, luôn về vị trí mới nhất
return -1 if ans == inf else ans
```

**Checklist:**
- [ ] So với lần xuất hiện **gần nhất** (cập nhật `last` mỗi bước).
- [ ] So sánh trước, cập nhật sau.
- [ ] `+ 1` để ra số lá.
- [ ] Không có cặp ⇒ `-1` (kiểu `int`).
- [ ] Đã thử: một lá, hai lá bằng nhau, giá trị xuất hiện ≥ 3 lần.